In [1]:
# Built-in
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, TimeDistributed, GlobalAveragePooling2D, LSTM, Dense, Dropout
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)


In [2]:
# Define the path to the dataset
base_path = '/Users/admin/AIEngineer/ai-deepfake-cv-project/Dataset/Faceplus2'
categories = ['fake', 'real']

# Initialize a list to hold data
data = []

# Process each category
for category in categories:
    category_path = os.path.join(base_path, category)
    for filename in os.listdir(category_path):
        if filename.endswith('.jpg'):
            try:
                id_part, frame_part = filename.split('_frame_')
                id_ = id_part.split('_')[0]
                frame = frame_part.split('.')[0]
                data.append({
                    'filename': filename,
                    'path': os.path.join(category_path, filename),
                    'id': int(id_),
                    'frame': int(frame),
                    'label': category
                })
            except ValueError:
                continue

# Convert the data to a DataFrame
df = pd.DataFrame(data)

In [3]:
# Đảm bảo đã có df_cropped.csv chứa đường dẫn ảnh đã crop
df['label_id'] = df['label'].map({'fake': 0, 'real': 1})
df


,filename,path,id,frame,label,label_id
0,596_609_frame_0019.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,596,19,fake,0
1,746_571_frame_0008.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,746,8,fake,0
2,514_443_frame_0028.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,514,28,fake,0
3,642_635_frame_0006.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,642,6,fake,0
4,937_888_frame_0009.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,937,9,fake,0
...,...,...,...,...,...,...
59803,727_frame_0009.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,727,9,real,1
59804,832_frame_0026.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,832,26,real,1
59805,059_frame_0005.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,59,5,real,1
59806,572_frame_0008.jpg,/Users/admin/AIEngineer/ai-deepfake-cv-project...,572,8,real,1


In [4]:
df['video_key'] = df['id'].astype(str) + "_" + df['label']

from collections import defaultdict

video_dict = defaultdict(list)
labels = {}

for _, row in df.iterrows():
    key = row['video_key']
    video_dict[key].append(row['path'])
    labels[key] = row['label_id']


In [5]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(),"optimizer")))

from adamax_sgd_mix_new import AdamaxSGDMix, AlphaAnneal

In [6]:
import tensorflow as tf, keras, platform
print("arch:", platform.machine())
print("tf:", tf.__version__)
print("keras:", keras.__version__)
print("devices:", tf.config.list_physical_devices())

arch: arm64
tf: 2.15.1
keras: 2.15.0
devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, TimeDistributed, LSTM, Dropout, Dense, GlobalAveragePooling2D, BatchNormalization, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from hmmlearn.hmm import GaussianHMM
from tfkan.layers import Conv2DKAN, DenseKAN  # Import KAN layers
from scipy.stats import mode
from tensorflow.keras.applications import EfficientNetB0


# Giả định đã có
video_keys = list(video_dict.keys())
video_labels = [labels[k] for k in video_keys]

img_size = (224, 224)
batch_size = 32
epochs = 50
n_splits = 5
sequence_len = 10
results = []
all_histories = []

import numpy as np, cv2, random

class VideoSequence(tf.keras.utils.Sequence):
    def __init__(self, video_keys, video_dict, labels, batch_size, img_size, sequence_len=16, augment=False, shuffle=True):
        self.video_keys = list(video_keys)
        self.video_dict = video_dict
        self.labels = labels
        self.batch_size = batch_size
        self.img_size = img_size
        self.sequence_len = sequence_len
        self.augment = augment
        self.shuffle = shuffle
        if self.shuffle:
            np.random.shuffle(self.video_keys)

    def __len__(self):
        return int(np.ceil(len(self.video_keys) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.video_keys)

    def _sample_frames(self, frame_paths):
        n = len(frame_paths)
        if n == 0:
            return [None]*self.sequence_len
        if n >= self.sequence_len:
            idx = np.linspace(0, n-1, self.sequence_len).astype(int)
        else:
            # lặp lại để đủ length
            reps = int(np.ceil(self.sequence_len / n))
            tiled = (np.arange(n).tolist()*reps)[:self.sequence_len]
            idx = np.array(tiled)
        return [frame_paths[i] if frame_paths else None for i in idx]

    def _augment_clip_params(self):
        # một bộ params áp cho toàn clip
        params = {
            "hflip": self.augment and (random.random() < 0.5),
            "rot_deg": random.uniform(-10, 10) if self.augment else 0.0,
            "alpha": 1.0 + (random.uniform(-0.1, 0.1) if self.augment else 0.0),  # contrast
            "beta": random.uniform(-10, 10) if self.augment else 0.0,             # brightness
        }
        return params

    def _apply_aug(self, img, params):
        if params["hflip"]:
            img = cv2.flip(img, 1)
        if abs(params["rot_deg"]) > 1e-3:
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w/2, h/2), params["rot_deg"], 1.0)
            img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        img = cv2.convertScaleAbs(img, alpha=params["alpha"], beta=params["beta"])
        return img

    def __getitem__(self, idx):
        batch_keys = self.video_keys[idx*self.batch_size:(idx+1)*self.batch_size]
        X, y = [], []
        for k in batch_keys:
            paths = self.video_dict[k]
            sel = self._sample_frames(paths)
            params = self._augment_clip_params()
            clip = []
            for p in sel:
                if p is None:
                    img = np.zeros((*self.img_size, 3), dtype=np.uint8)
                else:
                    img = cv2.imread(p)
                    img = cv2.resize(img, self.img_size, interpolation=cv2.INTER_AREA)
                img = self._apply_aug(img, params) if self.augment else img
                img = img.astype(np.float32)/255.0
                clip.append(img)
            X.append(clip)
            y.append(self.labels[k])
        return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)

# Build model cải tiến
def build_model(sequence_len, img_size):
    base_cnn = MobileNetV2(input_shape=(*img_size, 3), include_top=False, weights='imagenet')

    # Freeze toàn bộ backbone để giảm overfit
    base_cnn.trainable = False

    # CNN feature extractor
    cnn_out = GlobalAveragePooling2D()(base_cnn.output)
    cnn_model = Model(inputs=base_cnn.input, outputs=cnn_out)

    # Sequence input
    input_seq = Input(shape=(sequence_len, *img_size, 3))
    x = TimeDistributed(cnn_model)(input_seq)

    # Temporal modeling
    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.5)(x)

    # Classification
    x = DenseKAN(1)(x)
    output = tf.keras.activations.sigmoid(x)

    model = Model(inputs=input_seq, outputs=output)
    return model

# HMM cải tiến
def hmm_postprocess(pred_probs, y_true, n_states=2):
    pred_probs = pred_probs.reshape(-1, 1)
    hmm = GaussianHMM(n_components=n_states, covariance_type="diag", n_iter=100)
    hmm.fit(pred_probs)
    hidden_states = hmm.predict(pred_probs)

    mapping = {}
    for state in np.unique(hidden_states):
        indices = [i for i in range(len(hidden_states)) if hidden_states[i] == state]
        state_labels = [y_true[i] for i in indices]
        if len(state_labels) > 0:
            mapped_label = mode(state_labels, keepdims=True).mode[0]
        else:
            mapped_label = 0  # fallback
        mapping[state] = mapped_label

    hmm_labels = np.array([mapping[s] for s in hidden_states])
    return hmm_labels

# Training K-Fold
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold, (trainval_idx, test_idx) in enumerate(skf.split(video_keys, video_labels), 1):
    print(f"\n===== Fold {fold} =====")

    trainval_keys = [video_keys[i] for i in trainval_idx]
    test_keys = [video_keys[i] for i in test_idx]

    y_trainval = [labels[k] for k in trainval_keys]
    train_keys, val_keys = train_test_split(trainval_keys, test_size=0.1, stratify=y_trainval, random_state=fold)

    train_gen = VideoSequence(train_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=True)
    val_gen = VideoSequence(val_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=False)
    test_gen = VideoSequence(test_keys, video_dict, labels, batch_size, img_size, sequence_len, augment=False)

    model = build_model(sequence_len, img_size)

    mix_opt = AdamaxSGDMix(
        lr_adamax=1e-4,
        lr_sgd=5e-4,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7,
        momentum=0.9,
        nesterov=True,
        alpha=0.8
    )

    model.compile(optimizer=mix_opt, loss='binary_crossentropy', metrics=['accuracy'])

    alpha_cb = AlphaAnneal(mix_opt, start=0.8, end=0.1, mode="cosine")



    model_path = f"best_model_fold{fold}.h5"
    checkpoint = ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    earlystop = EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=[checkpoint, earlystop, reduce_lr, alpha_cb],
        verbose=1
    )

    all_histories.append(history.history)

    model.load_weights(model_path)

    y_true = [labels[k] for k in test_keys]
    y_pred_prob = model.predict(test_gen).ravel()
    y_hmm_pred = hmm_postprocess(y_pred_prob, y_true)

    results.append({
        'fold': fold,
        'accuracy': accuracy_score(y_true, y_hmm_pred),
        'precision': precision_score(y_true, y_hmm_pred),
        'recall': recall_score(y_true, y_hmm_pred),
        'f1': f1_score(y_true, y_hmm_pred),
        'auc': roc_auc_score(y_true, y_pred_prob)
    })

print("\n📊 Tổng kết kết quả các fold:")
for r in results:
    print(f"Fold {r['fold']}: Accuracy={r['accuracy']:.4f}, F1={r['f1']:.4f}, AUC={r['auc']:.4f}")


===== Fold 1 =====


Epoch 1/50
45/45 [==============================] - ETA: 0s - loss: 0.7048 - accuracy: 0.5083  
Epoch 1: val_accuracy improved from -inf to 0.58750, saving model to best_model_fold1.h5


/opt/anaconda3/envs/environment_tf_216/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


45/45 [==============================] - 51s 1s/step - loss: 0.7048 - accuracy: 0.5083 - val_loss: 0.6769 - val_accuracy: 0.5875 - lr: 1.0000
Epoch 2/50
45/45 [==============================] - ETA: 0s - loss: 0.6765 - accuracy: 0.5667 
Epoch 2: val_accuracy improved from 0.58750 to 0.63125, saving model to best_model_fold1.h5
45/45 [==============================] - 45s 993ms/step - loss: 0.6765 - accuracy: 0.5667 - val_loss: 0.6649 - val_accuracy: 0.6313 - lr: 1.0000
Epoch 3/50
45/45 [==============================] - ETA: 0s - loss: 0.6785 - accuracy: 0.5660 
Epoch 3: val_accuracy improved from 0.63125 to 0.65625, saving model to best_model_fold1.h5
45/45 [==============================] - 42s 927ms/step - loss: 0.6785 - accuracy: 0.5660 - val_loss: 0.6584 - val_accuracy: 0.6562 - lr: 1.0000
Epoch 4/50
45/45 [==============================] - ETA: 0s - loss: 0.6698 - accuracy: 0.6014 
Epoch 4: val_accuracy improved from 0.65625 to 0.67500, saving model to best_model_fold1.h5
45/45 [

Epoch 1/50
45/45 [==============================] - ETA: 0s - loss: 0.7075 - accuracy: 0.5049  
Epoch 1: val_accuracy improved from -inf to 0.56875, saving model to best_model_fold2.h5


...

45/45 [==============================] - 67s 1s/step - loss: 0.7075 - accuracy: 0.5049 - val_loss: 0.6862 - val_accuracy: 0.5688 - lr: 1.0000
Epoch 2/50
45/45 [==============================] - ETA: 0s - loss: 0.6857 - accuracy: 0.5354 
Epoch 2: val_accuracy improved from 0.56875 to 0.57500, saving model to best_model_fold2.h5
45/45 [==============================] - 78s 2s/step - loss: 0.6857 - accuracy: 0.5354 - val_loss: 0.6777 - val_accuracy: 0.5750 - lr: 1.0000
Epoch 3/50
45/45 [==============================] - ETA: 0s - loss: 0.6832 - accuracy: 0.5618  
Epoch 3: val_accuracy improved from 0.57500 to 0.61875, saving model to best_model_fold2.h5
45/45 [==============================] - 54s 1s/step - loss: 0.6832 - accuracy: 0.5618 - val_loss: 0.6724 - val_accuracy: 0.6187 - lr: 1.0000
Epoch 4/50
45/45 [==============================] - ETA: 0s - loss: 0.6694 - accuracy: 0.5847 
Epoch 4: val_accuracy did not improve from 0.61875
45/45 [==============================] - 54s 1s/step 

Epoch 1/50
45/45 [==============================] - ETA: 0s - loss: 0.6990 - accuracy: 0.5111  
Epoch 1: val_accuracy improved from -inf to 0.55000, saving model to best_model_fold3.h5


/opt/anaconda3/envs/environment_tf_216/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


45/45 [==============================] - 85s 2s/step - loss: 0.6990 - accuracy: 0.5111 - val_loss: 0.6832 - val_accuracy: 0.5500 - lr: 1.0000
Epoch 2/50
45/45 [==============================] - ETA: 0s - loss: 0.6773 - accuracy: 0.5771  
Epoch 2: val_accuracy improved from 0.55000 to 0.58750, saving model to best_model_fold3.h5
45/45 [==============================] - 71s 2s/step - loss: 0.6773 - accuracy: 0.5771 - val_loss: 0.6717 - val_accuracy: 0.5875 - lr: 1.0000
Epoch 3/50
45/45 [==============================] - ETA: 0s - loss: 0.6686 - accuracy: 0.5938  
Epoch 3: val_accuracy improved from 0.58750 to 0.61250, saving model to best_model_fold3.h5
45/45 [==============================] - 67s 1s/step - loss: 0.6686 - accuracy: 0.5938 - val_loss: 0.6649 - val_accuracy: 0.6125 - lr: 1.0000
Epoch 4/50
45/45 [==============================] - ETA: 0s - loss: 0.6528 - accuracy: 0.6153 
Epoch 4: val_accuracy improved from 0.61250 to 0.61875, saving model to best_model_fold3.h5
45/45 [====

AttributeError: 'float' object has no attribute 'dtype'

In [ ]:
results_df = pd.DataFrame(results)
print("📊 Kết quả trung bình:")
print(results_df.mean(numeric_only=True))
results_df


In [ ]:
for i, hist in enumerate(all_histories, 1):
    plt.figure()
    plt.plot(hist['accuracy'], label='Train Acc')
    plt.plot(hist['val_accuracy'], label='Val Acc')
    plt.title(f'Fold {i} Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure()
    plt.plot(hist['loss'], label='Train Loss')
    plt.plot(hist['val_loss'], label='Val Loss')
    plt.title(f'Fold {i} Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
# Save mô hình fold cuối cùng
model.save("mobilenetv2_faceplus_final.h5")


In [ ]:
import pandas as pd

# Giả sử results đã có và bạn đã tạo results_df
results_df = pd.DataFrame(results)

# Tính các chỉ số
accuracy_mean = results_df['accuracy'].mean()
accuracy_std = results_df['accuracy'].std()  # dùng sample std (chia cho n-1)
accuracy_range = results_df['accuracy'].max() - results_df['accuracy'].min()
accuracy_cv_percent = (accuracy_std / accuracy_mean) * 100

# In kết quả
print("📊 Kết quả trung bình:")
print(results_df.mean(numeric_only=True))

print(f"\n✅ CV Accuracy (Mean Accuracy): {accuracy_mean:.4f}")
print(f"📈 Range Accuracy: {accuracy_range:.4f}")
print(f"📉 Accuracy CV% (std/mean): {accuracy_cv_percent:.2f}%")

# Hiển thị bảng kết quả nếu cần
results_df
